# Phase 4C.7B: Independent Cohort Acquisition & Generation (Colab T4 GPU)

> **Document Class**: Google Colab Execution Launcher  
> **Phase**: Phase 4C.7B — Independent Cohort Acquisition & Feasibility Alignment  
> **Status**: `READY_FOR_COLAB_REAL_ACQUISITION_PILOT`  
> **Target Environment**: Google Colab (NVIDIA T4 GPU 15.0 GB VRAM or A100 GPU)  
> **Pinned Base Commit**: `4ea65df7ef415b2df41f0b3198bdbc264a7b5023` (branch `research/independent-cohort-acquisition`)  

---

## 1. Scientific Protocol & Immutable Invariants (Amendment v1.2)
- **Target Sample Size**: Exactly **400 source pairs** (800 images: 400 authentic, 400 ai_edited) + 400 request masks.
- **Orthogonal Balanced 2x2 Matrix** (100 pairs target per cell, 110 buffer pool per cell):
  1. `coco_sd2`: COCO 2017 Clean x Stable Diffusion 2 Inpainting (100 pairs)
  2. `coco_sdxl`: COCO 2017 Clean x SDXL Inpainting 1.0 (100 pairs)
  3. `unsplash_sd2`: Unsplash Pre-2022 x Stable Diffusion 2 Inpainting (100 pairs)
  4. `unsplash_sdxl`: Unsplash Pre-2022 x SDXL Inpainting 1.0 (100 pairs)
- **Authentic Photographs Only**: 100% real photographic sources with verified provenance (official COCO test2017 annotations and Unsplash Lite pre-2022). Zero synthetic/random noise arrays as authentic samples.
- **Fail-Closed Downloader**: Download failure discards candidate and selects next in-stratum buffer candidate. Zero fallback to placeholder pixels.
- **Multi-Dimensional Quota Preservation**: Replacement preserves exact strata (100 each), modification types (40% replacement, 30% removal, 30% insertion), and mask areas (30% small, 40% medium, 30% large).
- **Detector Isolation**: ZERO detector calls or scoring. Completely blind acquisition and technical QC.
- **Durable Resume**: All attempts and receipts are written to disk immediately with SHA-256 tamper verification.
- **Execution Policy**: Mặc định chạy Technical Pilot (2 pairs/stratum = 8 pairs total). Full chỉ chạy sau khi pilot PASS và content QC contact sheet được người dùng thẩm định.

In [ ]:
# Cell 1: Mount Google Drive & Environment Verification
import os, sys, shutil
from pathlib import Path

print("=== Step 1: Mounting Google Drive ===")
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_AVAILABLE = True
    DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/forensics-web-lab/phase_4c7b_cohort')
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted. Output directory: {DRIVE_OUTPUT_DIR}")
except Exception as e:
    print(f"Notice: Running in non-Colab or Drive mount skipped: {e}")
    DRIVE_AVAILABLE = False
    DRIVE_OUTPUT_DIR = Path('/content/phase_4c7b_cohort')
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\n=== Step 2: GPU Capability Check ===")
import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
    if vram_gb < 11.0:
        print("WARNING: VRAM < 12GB. Execution may require sequential CPU offload.")
else:
    print("WARNING: CUDA not detected! Go to Runtime -> Change runtime type -> T4 GPU.")

# Install required packages
!pip install -q diffusers transformers accelerate safetensors torchvision pillow requests

In [ ]:
# Cell 2: Checkout Repository & Verify Scientific Commit Binding
REPO_DIR = Path('/content/forensics-web-lab')
PINNED_BRANCH = "research/independent-cohort-acquisition"
PINNED_COMMIT = "4ea65df7ef415b2df41f0b3198bdbc264a7b5023"

if not REPO_DIR.is_dir():
    print(f"Cloning repository on branch {PINNED_BRANCH}...")
    !git clone --branch {PINNED_BRANCH} https://github.com/nomozer/forensics-web-lab.git {REPO_DIR}
else:
    print("Repository already exists. Fetching latest branch ref...")
    !cd {REPO_DIR} && git fetch origin {PINNED_BRANCH}

# Check status and current commit
print("\nRepository commit verification:")
!cd {REPO_DIR} && git status --short && git log -1 --oneline

# Add repository root to python sys.path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Verify candidate plan quotas pre-run
!cd {REPO_DIR} && python scripts/research/run_cohort_acquisition.py --verify-plan

In [ ]:
# Cell 3: Execution (Default: Technical Pilot Mode)
# Pilot runs 2 pairs per stratum (8 pairs total) on verified real photos
# Measures latency, VRAM, and outputs visual contact sheet before full acquisition.

PILOT_OUT = DRIVE_OUTPUT_DIR / "pilot_cohort"
PILOT_OUT.mkdir(parents=True, exist_ok=True)

print(f">>> Executing Technical Pilot on Colab GPU (Output: {PILOT_OUT}) ...")
!cd {REPO_DIR} && python scripts/research/run_cohort_acquisition.py \
    --mode pilot \
    --output-dir "{PILOT_OUT}" \
    --contact-sheet

# ==========================================================================
# FULL ACQUISITION RUN (400 pairs / 800 images):
# Run ONLY after Pilot completes, Technical QC passes, and Content QC is reviewed.
# Uncomment the following lines to execute the full acquisition run:
#
# FULL_OUT = DRIVE_OUTPUT_DIR / "full_cohort"
# FULL_OUT.mkdir(parents=True, exist_ok=True)
# !cd {REPO_DIR} && python scripts/research/run_cohort_acquisition.py \
#     --mode full \
#     --output-dir "{FULL_OUT}" \
#     --contact-sheet
# ==========================================================================

In [ ]:
# Cell 4: Verify Artifacts, Content QC Preview & Packaging
import json, csv, hashlib
from IPython.display import display, HTML

RUN_OUT = PILOT_OUT  # Change to FULL_OUT when running full acquisition
MANIFEST_FILE = RUN_OUT / "manifest_independent_cohort.csv"
CHECKSUM_FILE = RUN_OUT / "manifest_checksum.sha256"
ATTEMPT_LEDGER = RUN_OUT / "attempt_ledger.jsonl"
PROVENANCE_LEDGER = RUN_OUT / "provenance_ledger.jsonl"
CONTACT_SHEET = RUN_OUT / "content_qc_contact_sheet.html"

print("=== Acquisition Run Audit ===")
if MANIFEST_FILE.is_file() and CHECKSUM_FILE.is_file():
    calc_sha = hashlib.sha256(MANIFEST_FILE.read_bytes()).hexdigest()
    recorded_line = CHECKSUM_FILE.read_text().strip()
    recorded_sha = recorded_line.split()[0]
    assert calc_sha == recorded_sha, f"Checksum mismatch: {calc_sha} vs {recorded_sha}"
    
    with MANIFEST_FILE.open('r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    print(f"Manifest Status: VALID (SHA-256: {calc_sha})")
    print(f"Total Images: {len(rows)} ({len(rows)//2} source pairs)")
    
    if ATTEMPT_LEDGER.is_file():
        attempts = [json.loads(l) for l in ATTEMPT_LEDGER.read_text().splitlines() if l.strip()]
        print(f"Attempt Ledger: {len(attempts)} recorded attempts on disk")
    if PROVENANCE_LEDGER.is_file():
        receipts = [json.loads(l) for l in PROVENANCE_LEDGER.read_text().splitlines() if l.strip()]
        print(f"Provenance Ledger: {len(receipts)} accepted sample receipts on disk")
    
    # Package archive for local download
    ZIP_OUT = DRIVE_OUTPUT_DIR / f"{RUN_OUT.name}_package"
    shutil.make_archive(str(ZIP_OUT), 'zip', RUN_OUT)
    print(f"Packaged ZIP archive in Drive: {ZIP_OUT}.zip")
    
    if CONTACT_SHEET.is_file():
        print(f"Content QC Contact Sheet generated at: {CONTACT_SHEET}")
else:
    print("Manifest not found. Run has not completed yet.")